# IN4640 Assignment 2 — Fitting and Alignment
**Student Name:** [Your Name Here]  
**Index Number:** [Your Index Here]

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import pandas as pd
from sklearn.linear_model import RANSACRegressor, LinearRegression

---
## Question 1 — Total Least Squares & RANSAC Line Fitting

### 1(a) Total Least Squares on Line 1 Only

In [ ]:
# Load dataset
D = np.genfromtxt('lines.csv', delimiter=',', skip_header=1)
print('Dataset shape:', D.shape)
print('First 5 rows:')
pd.DataFrame(D, columns=['x1','x2','x3','y1','y2','y3']).head()

In [ ]:
# Extract line 1 data (columns x1, y1)
x1 = D[:, 0]
y1 = D[:, 3]

# Total Least Squares (TLS) using SVD
# Fit line: ax + by + c = 0  (homogeneous form)
# Center the data first
mean_x = np.mean(x1)
mean_y = np.mean(y1)

# Build the data matrix
A = np.column_stack([x1 - mean_x, y1 - mean_y])

# SVD
U, S, Vt = np.linalg.svd(A)

# The last right singular vector gives the normal to the line
a, b = Vt[-1]   # normal vector (a, b)

# c from: a*mean_x + b*mean_y + c = 0
c = -(a * mean_x + b * mean_y)

print(f'TLS Line Parameters (ax + by + c = 0):')
print(f'  a = {a:.6f}')
print(f'  b = {b:.6f}')
print(f'  c = {c:.6f}')

# Convert to slope-intercept form: y = mx + k
if abs(b) > 1e-10:
    m = -a / b
    k = -c / b
    print(f'\nSlope-intercept form: y = {m:.6f}x + {k:.6f}')

In [ ]:
# Plot TLS fit for line 1
x_plot = np.linspace(x1.min(), x1.max(), 200)
y_plot = m * x_plot + k

plt.figure(figsize=(7, 5))
plt.scatter(x1, y1, s=15, label='Line 1 data', alpha=0.7)
plt.plot(x_plot, y_plot, 'r-', linewidth=2, label=f'TLS Fit: y={m:.3f}x+{k:.3f}')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Q1(a): Total Least Squares Fit — Line 1')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('q1a_tls_fit.png', dpi=150)
plt.show()
print('Figure saved as q1a_tls_fit.png')

### 1(b) RANSAC — Fit Three Lines Using All Points

In [ ]:
# Flatten all columns as instructed
X_cols = D[:, :3]
Y_cols = D[:, 3:]
X_all = X_cols.flatten()
Y_all = Y_cols.flatten()

print(f'Total points: {len(X_all)}')

def ransac_tls_line(x, y, residual_threshold=0.5, max_trials=1000):
    """Run RANSAC and return inlier mask + TLS parameters."""
    X_col = x.reshape(-1, 1)
    ransac = RANSACRegressor(
        estimator=LinearRegression(),
        residual_threshold=residual_threshold,
        max_trials=max_trials,
        random_state=42
    )
    ransac.fit(X_col, y)
    inlier_mask = ransac.inlier_mask_

    # Refit using TLS on inliers
    xi, yi = x[inlier_mask], y[inlier_mask]
    mx, my = np.mean(xi), np.mean(yi)
    A = np.column_stack([xi - mx, yi - my])
    _, _, Vt = np.linalg.svd(A)
    a, b = Vt[-1]
    c = -(a * mx + b * my)
    slope = -a / b if abs(b) > 1e-10 else np.inf
    intercept = -c / b if abs(b) > 1e-10 else np.inf
    return inlier_mask, (a, b, c), (slope, intercept)


remaining_x = X_all.copy()
remaining_y = Y_all.copy()
remaining_idx = np.arange(len(X_all))

lines = []
colors = ['red', 'blue', 'green']

for i in range(3):
    mask, params_abc, params_mi = ransac_tls_line(remaining_x, remaining_y)
    a, b, c = params_abc
    slope, intercept = params_mi
    n_inliers = mask.sum()
    print(f'Line {i+1}: a={a:.4f}, b={b:.4f}, c={c:.4f} | '
          f'y={slope:.4f}x+{intercept:.4f} | inliers={n_inliers}')
    lines.append({
        'x_inliers': remaining_x[mask],
        'y_inliers': remaining_y[mask],
        'abc': params_abc,
        'mi': params_mi
    })
    # Remove inliers (outliers become next iteration's data)
    remaining_x = remaining_x[~mask]
    remaining_y = remaining_y[~mask]

In [ ]:
# Plot all three RANSAC lines
plt.figure(figsize=(9, 6))
for i, (line, color) in enumerate(zip(lines, colors)):
    xi = line['x_inliers']
    yi = line['y_inliers']
    slope, intercept = line['mi']
    x_plot = np.linspace(xi.min(), xi.max(), 200)
    plt.scatter(xi, yi, s=12, color=color, alpha=0.6, label=f'Line {i+1} data')
    plt.plot(x_plot, slope * x_plot + intercept, color=color,
             linewidth=2, label=f'Line {i+1}: y={slope:.3f}x+{intercept:.3f}')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Q1(b): RANSAC + TLS — Three Lines')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('q1b_ransac_lines.png', dpi=150)
plt.show()
print('Figure saved as q1b_ransac_lines.png')

---
## Question 2 — Earring Size from Camera Parameters

In [ ]:
# Given camera parameters
f_mm = 8.0          # focal length in mm
pixel_size_um = 2.2 # pixel size in micrometres (2.2 µm)
Z_mm = 720.0        # distance from lens to imaging plane (mm)

# Convert pixel size to mm
pixel_size_mm = pixel_size_um / 1000.0  # 0.0022 mm

# Focal length in pixels
f_px = f_mm / pixel_size_mm
print(f'Focal length: {f_mm} mm = {f_px:.2f} pixels')

# Load earring image and measure in pixels
img_ear = cv2.imread('earrings.jpg')
h, w = img_ear.shape[:2]
print(f'Earring image size: {w} x {h} pixels')

# Convert to grayscale for measurements
gray = cv2.cvtColor(img_ear, cv2.COLOR_BGR2GRAY)

# Threshold to find earring extent
_, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY_INV)
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Measure each earring
img_annotated = img_ear.copy()
sizes_mm = []

# Sort contours by area (largest first)
contours = sorted(contours, key=cv2.contourArea, reverse=True)[:2]

for i, cnt in enumerate(contours):
    x, y, cw, ch = cv2.boundingRect(cnt)
    # Diameter in pixels (use average of width and height)
    diameter_px = (cw + ch) / 2.0

    # Real-world size using thin-lens formula:
    # size_real = size_pixels * pixel_size_mm * Z_mm / f_mm
    size_real_mm = diameter_px * pixel_size_mm * Z_mm / f_mm

    sizes_mm.append(size_real_mm)
    print(f'Earring {i+1}: {cw}x{ch} px | ~{diameter_px:.1f} px diameter | '
          f'Real size ≈ {size_real_mm:.2f} mm')

    cv2.rectangle(img_annotated, (x, y), (x+cw, y+ch), (0, 255, 0), 3)
    cv2.putText(img_annotated, f'{size_real_mm:.1f}mm',
                (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0,255,0), 2)

print(f'\nEstimated earring diameter: {np.mean(sizes_mm):.2f} mm (average)')

In [ ]:
# Display annotated earring image
plt.figure(figsize=(8, 6))
plt.imshow(cv2.cvtColor(img_annotated, cv2.COLOR_BGR2RGB))
plt.title('Q2: Earring Size Estimation')
plt.axis('off')
plt.tight_layout()
plt.savefig('q2_earring_size.png', dpi=150)
plt.show()
print('Figure saved as q2_earring_size.png')

---
## Question 3 — Superimpose Flag on Cricket Turf

### Step 1 — Interactive Corner Picker

Run the cell below. An image will appear — **click the 4 corners of the cricket pitch** in this order:

```
  1 ──────── 2
  |          |
  4 ──────── 3
```

- **Click 1:** Top-Left corner of the pitch
- **Click 2:** Top-Right corner of the pitch
- **Click 3:** Bottom-Right corner of the pitch
- **Click 4:** Bottom-Left corner of the pitch

After 4 clicks the selection locks automatically. Re-run the cell to reset.

In [ ]:
%matplotlib widget
# NOTE: requires   pip install ipympl
# If ipympl is unavailable fall back to:  %matplotlib notebook

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
from IPython.display import display
import ipywidgets as widgets

# ── Load images ──────────────────────────────────────────────────────────────
turf_img = cv2.imread('turf.jpg')
flag_img = cv2.imread('luxembourg_flag.jpg')
turf_rgb = cv2.cvtColor(turf_img, cv2.COLOR_BGR2RGB)
turf_h, turf_w = turf_img.shape[:2]
print(f'Turf : {turf_w} × {turf_h} px')
print(f'Flag : {flag_img.shape[1]} × {flag_img.shape[0]} px')

# ── State ────────────────────────────────────────────────────────────────────
clicked_pts   = []        # list of [x, y] in image pixels
CORNER_LABELS = ['1 (Top-Left)', '2 (Top-Right)',
                 '3 (Bottom-Right)', '4 (Bottom-Left)']
CORNER_COLORS = ['#00FF00', '#00FFFF', '#FF8800', '#FF00FF']
dot_artists   = []
line_artists  = []

# ── Build figure ─────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 7))
ax.imshow(turf_rgb)
ax.set_title('🖱️  Click the 4 corners of the cricket pitch  '
             '(Top-Left → Top-Right → Bottom-Right → Bottom-Left)',
             fontsize=11, pad=10)
ax.axis('off')

# Status label widget shown below the figure
status_lbl = widgets.HTML(
    value="<b style='color:navy'>👆 Click corner 1 (Top-Left)</b>")
undo_btn   = widgets.Button(description='↩ Undo Last',
                             button_style='warning', layout=widgets.Layout(width='130px'))
reset_btn  = widgets.Button(description='🔄 Reset All',
                             button_style='danger',  layout=widgets.Layout(width='130px'))
display(widgets.HBox([status_lbl, undo_btn, reset_btn]))

def redraw_overlay():
    """Clear and redraw all dots + connecting lines."""
    for a in dot_artists + line_artists:
        a.remove()
    dot_artists.clear()
    line_artists.clear()

    n = len(clicked_pts)
    for i, (px, py) in enumerate(clicked_pts):
        dot = ax.plot(px, py, 'o', color=CORNER_COLORS[i],
                      markersize=10, markeredgecolor='white',
                      markeredgewidth=1.5, zorder=5)[0]
        txt = ax.text(px + 12, py - 12, CORNER_LABELS[i],
                      color=CORNER_COLORS[i], fontsize=9,
                      fontweight='bold',
                      bbox=dict(boxstyle='round,pad=0.2',
                                fc='black', alpha=0.55, ec='none'),
                      zorder=6)
        dot_artists.extend([dot, txt])

    # Draw lines between selected points (close polygon when all 4 picked)
    pts_to_connect = clicked_pts + ([clicked_pts[0]] if n == 4 else [])
    for i in range(len(pts_to_connect) - 1):
        x0, y0 = pts_to_connect[i]
        x1, y1 = pts_to_connect[i + 1]
        ln = ax.plot([x0, x1], [y0, y1], '-',
                     color='red', linewidth=2, alpha=0.8, zorder=4)[0]
        line_artists.append(ln)

    fig.canvas.draw_idle()

def on_click(event):
    if event.inaxes is not ax:
        return
    if len(clicked_pts) >= 4:
        return          # locked after 4 clicks
    if event.button != 1:
        return          # left-click only

    ix, iy = int(round(event.xdata)), int(round(event.ydata))
    clicked_pts.append([ix, iy])
    print(f'  Point {len(clicked_pts)}: ({ix}, {iy})  — {CORNER_LABELS[len(clicked_pts)-1]}')
    redraw_overlay()

    if len(clicked_pts) < 4:
        status_lbl.value = (
            f"<b style='color:navy'>👆 Click corner {len(clicked_pts)+1} "
            f"({CORNER_LABELS[len(clicked_pts)]})</b>")
    else:
        status_lbl.value = (
            "<b style='color:green'>✅ All 4 corners selected! "
            "Scroll down and run the next cell to apply the flag.</b>")
        ax.set_title('✅ 4 corners locked — run the next cell to apply the flag',
                     fontsize=11, color='green', pad=10)
        fig.canvas.draw_idle()

def on_undo(_):
    if clicked_pts:
        removed = clicked_pts.pop()
        print(f'  Removed point: {removed}')
        redraw_overlay()
        status_lbl.value = (
            f"<b style='color:navy'>👆 Click corner {len(clicked_pts)+1} "
            f"({CORNER_LABELS[len(clicked_pts)]})</b>")
        ax.set_title('🖱️  Click the 4 corners of the cricket pitch  '
                     '(Top-Left → Top-Right → Bottom-Right → Bottom-Left)',
                     fontsize=11, pad=10)
        fig.canvas.draw_idle()

def on_reset(_):
    clicked_pts.clear()
    redraw_overlay()
    status_lbl.value = "<b style='color:navy'>👆 Click corner 1 (Top-Left)</b>"
    ax.set_title('🖱️  Click the 4 corners of the cricket pitch  '
                 '(Top-Left → Top-Right → Bottom-Right → Bottom-Left)',
                 fontsize=11, pad=10)
    fig.canvas.draw_idle()
    print('  Reset — all points cleared.')

# Wire up events
fig.canvas.mpl_connect('button_press_event', on_click)
undo_btn.on_click(on_undo)
reset_btn.on_click(on_reset)
plt.tight_layout()
plt.show()

### Step 2 — Apply Flag (run after selecting corners above)

In [ ]:
%matplotlib inline

# ── Validate corners ─────────────────────────────────────────────────────────
if len(clicked_pts) != 4:
    raise ValueError(
        f'Need exactly 4 corners — currently have {len(clicked_pts)}. '
        'Go back and run the picker cell again.')

dst_pts = np.float32(clicked_pts)   # [TL, TR, BR, BL]
print('Selected turf corners (dst_pts):')
for i, p in enumerate(dst_pts):
    print(f'  P{i+1} {CORNER_LABELS[i]}: ({int(p[0])}, {int(p[1])})')

# ── Homography ───────────────────────────────────────────────────────────────
fh, fw = flag_img.shape[:2]
src_pts = np.float32([
    [0,    0   ],   # → TL
    [fw-1, 0   ],   # → TR
    [fw-1, fh-1],   # → BR
    [0,    fh-1],   # → BL
])

H, status = cv2.findHomography(src_pts, dst_pts)
print('\nHomography matrix H:')
print(np.round(H, 6))

# ── Warp & blend ─────────────────────────────────────────────────────────────
flag_warped = cv2.warpPerspective(flag_img, H, (turf_w, turf_h))
mask = cv2.warpPerspective(
    np.ones((fh, fw), dtype=np.uint8) * 255, H, (turf_w, turf_h))

result = turf_img.copy()
result[mask > 0] = flag_warped[mask > 0]

# ── Annotate original with selected quad ─────────────────────────────────────
turf_vis = turf_img.copy()
pts_draw = dst_pts.astype(np.int32).reshape((-1, 1, 2))
cv2.polylines(turf_vis, [pts_draw], isClosed=True, color=(0, 0, 255), thickness=3)
for i, p in enumerate(dst_pts.astype(int)):
    clr = [(0,255,0),(0,255,255),(255,128,0),(255,0,255)][i]
    cv2.circle(turf_vis, tuple(p), 10, clr, -1)
    cv2.putText(turf_vis, str(i+1), (p[0]+12, p[1]-8),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, clr, 2)

# ── Display ──────────────────────────────────────────────────────────────────
fig2, axes = plt.subplots(1, 2, figsize=(14, 7))

axes[0].imshow(cv2.cvtColor(turf_vis, cv2.COLOR_BGR2RGB))
axes[0].set_title('Turf with selected quadrangle', fontsize=12)
axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
axes[1].set_title('Q3: Luxembourg Flag on Cricket Turf', fontsize=12)
axes[1].axis('off')

plt.tight_layout()
plt.savefig('q3_flag_on_turf.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved → q3_flag_on_turf.png')

### (Optional) Step 3 — Re-pick corners without restarting kernel

If you're unhappy with the result, simply **go back to Step 1 and re-run that cell** — it resets the point list automatically. Then re-run Step 2.

---
## Summary of Results

| Question | Result |
|----------|--------|
| 1(a) TLS Line 1 | Reported above (a, b, c) and slope/intercept |
| 1(b) RANSAC x3  | Three lines found with parameters printed above |
| 2 Earring size  | Estimated diameter in mm printed above |
| 3 Flag overlay  | Luxembourg flag projected onto cricket pitch via interactive corner selection |